# Click-Through Rate Prediction

## Learning Objectives

1. **Formulate** CTR prediction as online logistic regression
2. **Implement** the hashing trick for unbounded feature spaces
3. **Apply** online gradient descent and FTRL updates
4. **Evaluate** with log-loss and AUC

## The CTR Prediction Problem

When a user searches for "running shoes", the ad system must predict:

$$p(\text{click} \mid \text{query, ad, user, context})$$

for thousands of candidate ads in < 50ms.

**Scale challenges:**
- Billions of (query, ad) feature pairs
- Must update model in real time as clicks arrive
- Model must be served at millions of QPS

**Solution:** sparse linear model with online learning + the hashing trick.

## Feature Engineering

Raw features: query tokens, ad tokens, user history tokens, time of day, device.

**Cross features:** pair query token with ad token → captures "query 'shoes' + ad 'Nike' → high CTR".
Without crossing: 10K query tokens + 10K ad tokens = 20K features.
With crossing: 10K × 10K = 100M features — too large for a dictionary.

**Hashing trick:** map feature string → integer index via hash(feature) % num_buckets.
- Fixed memory regardless of number of unique features
- Collisions cause small accuracy loss; use large enough table (2²⁴ buckets common)
- No need to maintain a feature vocabulary

In [1]:
import math, random

class OnlineLR:
    """Sparse online logistic regression with hashing trick and SGD."""
    def __init__(self, n_bits=18, eta=0.1, l2=1e-4):
        self.n = 1 << n_bits          # number of buckets = 2^n_bits
        self.w = {}                    # sparse weight dict
        self.eta = eta
        self.l2 = l2

    def _hash(self, feature):
        return hash(feature) % self.n

    def predict(self, features):
        """features: list of string feature names (already one-hot / binary)."""
        dot = sum(self.w.get(self._hash(f), 0.0) for f in features)
        return 1.0 / (1.0 + math.exp(-dot))   # sigmoid

    def update(self, features, y):
        p = self.predict(features)
        err = p - y
        for f in features:
            idx = self._hash(f)
            w_i = self.w.get(idx, 0.0)
            self.w[idx] = w_i - self.eta * (err + self.l2 * w_i)
        return -math.log(p + 1e-10) if y == 1 else -math.log(1 - p + 1e-10)

def make_features(query_tokens, ad_tokens, device):
    """Build feature list including cross-features."""
    feats = []
    feats += [f'q:{t}' for t in query_tokens]
    feats += [f'a:{t}' for t in ad_tokens]
    feats += [f'dev:{device}']
    feats += [f'cross:{q}_{a}' for q in query_tokens for a in ad_tokens]
    return feats

# Demo
model = OnlineLR(n_bits=16, eta=0.05)
random.seed(42)

# Synthetic data: "running" + "shoes" → high CTR; "weather" + "shoes" → low CTR
def simulate_click(query, ad):
    if 'running' in query and 'shoes' in ad: return random.random() < 0.3
    if 'running' in query and 'jacket' in ad: return random.random() < 0.1
    return random.random() < 0.05

queries = [['running','marathon'], ['weather','forecast'], ['running','shoes'], ['buy','jacket']]
ads = [['shoes','nike'], ['jacket','adidas'], ['shoes','sale'], ['weather','app']]
devices = ['mobile','desktop']

losses = []
for step in range(500):
    q = random.choice(queries); a = random.choice(ads); dev = random.choice(devices)
    feats = make_features(q, a, dev)
    y = int(simulate_click(q, a))
    loss = model.update(feats, y)
    losses.append(loss)

# Show last 100 predictions
correct = 0; total = 50
for _ in range(total):
    q = random.choice(queries); a = random.choice(ads); dev = random.choice(devices)
    feats = make_features(q, a, dev)
    p = model.predict(feats)
    y = int(simulate_click(q, a))
    correct += int((p > 0.15) == y)
print(f"Avg log-loss (last 100 steps): {sum(losses[-100:])/100:.4f}")
print(f"Accuracy on 50 held-out: {correct}/{total}")

Avg log-loss (last 100 steps): 0.2816
Accuracy on 50 held-out: 39/50


## FTRL: Follow-The-Regularized-Leader

SGD with a fixed learning rate doesn't work well for sparse features — features seen rarely get the same step size as common features.

**FTRL-Proximal** (McMahan et al., Google 2013) uses per-coordinate adaptive learning rates:

$$w_i = -\frac{1}{\lambda_2 + \beta_i / \eta}\left(z_i - \text{sign}(z_i)\lambda_1\right) \quad \text{if } |z_i| > \lambda_1$$

where $z_i$ accumulates past gradients and $\beta_i = \sqrt{\sum_{t} g_{t,i}^2}$ (AdaGrad-like).

**Benefits:**
- Per-coordinate learning rates: rare features get large steps
- $L_1$ regularization produces sparse models (many weights = 0)
- Outperforms SGD significantly on CTR prediction tasks

In [2]:
import math

class FTRL:
    """FTRL-Proximal with L1/L2 regularization (McMahan et al. 2013)."""
    def __init__(self, n_bits=18, alpha=0.1, beta=1.0, l1=0.1, l2=1e-4):
        self.n = 1 << n_bits
        self.alpha = alpha; self.beta = beta; self.l1 = l1; self.l2 = l2
        self.z = {}   # accumulated gradient stats
        self.n_sq = {}   # sum of squared gradients

    def _w(self, i):
        z_i = self.z.get(i, 0.0)
        if abs(z_i) <= self.l1:
            return 0.0
        n_i = self.n_sq.get(i, 0.0)
        return -(z_i - math.copysign(self.l1, z_i)) / (self.l2 + (self.beta + math.sqrt(n_i)) / self.alpha)

    def predict(self, features):
        dot = sum(self._w(hash(f) % self.n) for f in features)
        return 1.0 / (1.0 + math.exp(-dot))

    def update(self, features, y):
        p = self.predict(features)
        for f in features:
            i = hash(f) % self.n
            g = p - y
            n_i = self.n_sq.get(i, 0.0)
            sigma = (math.sqrt(n_i + g*g) - math.sqrt(n_i)) / self.alpha
            self.z[i] = self.z.get(i, 0.0) + g - sigma * self._w(i)
            self.n_sq[i] = n_i + g*g
        return -math.log(p+1e-10) if y==1 else -math.log(1-p+1e-10)

model_ftrl = FTRL(n_bits=16, alpha=0.1, l1=0.05)
losses_ftrl = []
for step in range(500):
    q = random.choice(queries); a = random.choice(ads); dev = random.choice(devices)
    feats = make_features(q, a, dev)
    y = int(simulate_click(q, a))
    losses_ftrl.append(model_ftrl.update(feats, y))

sparsity = sum(1 for w in model_ftrl.z.values() if abs(w) <= model_ftrl.l1) / max(len(model_ftrl.z),1)
print(f"FTRL avg log-loss (last 100): {sum(losses_ftrl[-100:])/100:.4f}")
print(f"SGD  avg log-loss (last 100): {sum(losses[-100:])/100:.4f}")
print(f"FTRL model sparsity: {sparsity:.1%} of active weights are zeroed by L1")

FTRL avg log-loss (last 100): 0.2939
SGD  avg log-loss (last 100): 0.2816
FTRL model sparsity: 0.0% of active weights are zeroed by L1


## Calibration

Raw logistic output may not be well-calibrated: $p = 0.3$ should mean "30% of similar (query,ad) pairs get clicked."

**Platt scaling:** fit a sigmoid $q = \sigma(a \cdot p + b)$ on a held-out calibration set.

**Isotonic regression:** fit a non-decreasing step function — more flexible than Platt.

**Why calibration matters:** ad systems use predicted CTR to compute expected value (bid × CTR); miscalibration → suboptimal allocation.

## Summary

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Model | Sparse logistic regression | Interpretable; fast to serve |
| Features | Cross-features via hashing | Handles 10⁸+ feature pairs |
| Training | FTRL-Proximal | Per-coord LR; L1 sparsity |
| Calibration | Platt / isotonic | CTR must be a true probability |

This is essentially the system described in Google's 2013 KDD paper "Ad Click Prediction: a View from the Trenches" — it powers billions of dollars of ad revenue using a simple linear model with careful engineering.